In [1]:
import pandas as pd
import numpy as np
import warnings;warnings.filterwarnings('ignore')


# display max row & columns
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None) 
pd.set_option('display.max_colwidth', None)

In [2]:
df = pd.read_excel(r"H:\CampusX_DS\week43 - My Projects Aug 2024\end-to-end-used-car-price-prediction\cars24_final_data.xlsx")

In [3]:
### cleaning

In [4]:
df.drop_duplicates(inplace=True)

df = df[df['specs_tag'] == "available"]
df = df[df[['Displacementcc', 'GearBoxNumberOfGears']].notna().all(axis=1)]
df.drop(columns=['TransmissionType'], inplace=True)

In [5]:
def is_valid_column(col):
    unique_values = set(col.dropna())  # Exclude NaN for checking unique values
    return unique_values.issubset({1.0, 0.0}) and \
           all(val in unique_values for val in [1.0, 0.0]) and \
           col.isnull().any()  # Ensure at least one NaN is present

# Identify columns that meet the criteria
valid_columns = df.columns[df.apply(is_valid_column, axis=0)]

# Create a new DataFrame with only the valid columns
filtered_df = df[valid_columns]

df.drop(columns=filtered_df.columns.tolist(), inplace=True)

In [6]:
# Convert Unix timestamps to datetime
df['content.insuranceExpiry'] = pd.to_datetime(df['content.insuranceExpiry'], unit='s')

# Format the datetime to the desired string format
df['content.insuranceExpiry'] = df['content.insuranceExpiry'].dt.strftime('%d-%b-%Y')

# Convert the datetime strings to datetime objects
df['content.lastServicedAt'] = pd.to_datetime(df['content.lastServicedAt'])

# Format the datetime to the desired string format
df['content.lastServicedAt'] = df['content.lastServicedAt'].dt.strftime('%d-%b-%Y')

## removing filter column which is of no use in analysis & model building

df.drop(columns=['specs_tag'], inplace=True)

In [7]:
## change content.fitnessUpto, content.insuranceExpiry & content.lastServicedAt column to months from today

from datetime import datetime

# Convert to datetime
df['content.fitnessUpto'] = pd.to_datetime(df['content.fitnessUpto'], format='%d-%b-%Y')

# Get today's date
today = pd.to_datetime(datetime.now().date())

df['content.fitnessUpto_months_remaining'] = ((df['content.fitnessUpto'].dt.year - today.year) * 12 + 
                          (df['content.fitnessUpto'].dt.month - today.month))

from datetime import datetime

# Convert to datetime
df['content.insuranceExpiry'] = pd.to_datetime(df['content.insuranceExpiry'], format='%d-%b-%Y')

# Get today's date
today = pd.to_datetime(datetime.now().date())

df['content.insuranceExpiry_months_remaining'] = ((df['content.insuranceExpiry'].dt.year - today.year) * 12 + (df['content.insuranceExpiry'].dt.month - today.month))
df
## change all values less then 0 to 0
df['content.insuranceExpiry_months_remaining'] = df['content.insuranceExpiry_months_remaining'].clip(lower=0)

from datetime import datetime

# Convert to datetime
df['content.lastServicedAt'] = pd.to_datetime(df['content.lastServicedAt'], format='%d-%b-%Y')

# Get today's date
today = pd.to_datetime(datetime.now().date())

df['content.lastServicedAt_months_remaining'] = ((df['content.lastServicedAt'].dt.year - today.year) * 12 + (df['content.lastServicedAt'].dt.month - today.month))

## change all negative values to positive
df['content.lastServicedAt_months_remaining'] = df['content.lastServicedAt_months_remaining'].abs()

## removing columns as already been feature engineered

df.drop(columns=['content.fitnessUpto', 'content.insuranceExpiry','content.lastServicedAt'], inplace=True)

##remove content.city
df.drop(columns=['content.city'], inplace=True)

## extract state code from content.cityRto
df['content.staterto'] = df['content.cityRto'].str[:2]

df.drop(columns=['content.staterto'], inplace=True)

## change NaN to "not available" for selected columns

cols_to_replace = ['ABSAntilockBrakingSystem', 'PowerWindowsFront', 'PowerWindowsRear', 
                   'AirConditioner', '12VPowerOutlet', 'MultifunctionDisplayScreen', 
                   'EntertainmentDisplayScreen', 'VoiceRecognition', 'SmartCardSmartKey', 
                   'AmbientLighting', 'SunroofMoonroof', 'WirelessChargingPad', 
                   'VentilatedSeatsRear', 'HVACControl', '360DegreeCamera', 
                   'ParkingAssistSide', 'DriverSeatAdjustmentElectric']

df[cols_to_replace] = df[cols_to_replace].fillna("not available")


## missing values in HeadlampBulbTypeHighBeam can be filled using HeadlampBulbTypeLowBeam column
df['HeadlampBulbTypeHighBeam'] = df['HeadlampBulbTypeHighBeam'].fillna(df['HeadlampBulbTypeLowBeam'])

## strip whole dataset
df = df.applymap(lambda x: x.strip() if isinstance(x, str) else x)

## Change all not available to 0 for better visualization
## To ensure that the changes are applied only when the columns contain exactly two unique values as specified

def update_columns(df):
    for col in df.columns:
        unique_values = set(df[col].astype(str).unique())
#         print(f"Processing column: {col}, Unique values: {unique_values}")  # Debugging line
        
        if unique_values == {'1', 'not available'}:
            df[col] = df[col].replace('not available', 0)
        
        elif unique_values == {'Available', 'Not available'}:
            df[col] = df[col].replace({'Available': 1, 'Not available': 0})
            
        elif unique_values == {'0', 'not available'}:
            df[col] = df[col].replace('not available', 0)
            
# Apply the function to your DataFrame
update_columns(df)


## Change content.duplicatekey True/False to 1/0
df['content.duplicateKey'] = df['content.duplicateKey'].replace({True: 1, False: 0})


## Extract State from cityrto
df['car_state'] = df['content.cityRto'].str.strip().str[:2]


## content.cityRto content.registrationNumber content.listingPrice - remove
df.drop(columns=['content.cityRto','content.registrationNumber','content.listingPrice'], inplace=True)


df.drop(columns=['content.appointmentId'], inplace=True)



# Replace "OK" with 1 and "WARN" with 0 in specified columns
columns_to_replace = ['Left Front Tyre', 'Right Front Tyre', 'Left Rear Tyre', 'Right Rear Tyre', 'Spare Tyre']
df[columns_to_replace] = df[columns_to_replace].replace({'OK': 1, 'WARN': 0})


# Calculate the sum of 'OK' values
df['Tyre_Health'] = df[['Left Front Tyre', 'Right Front Tyre', 'Left Rear Tyre', 'Right Rear Tyre', 'Spare Tyre']].sum(axis=1)

# Calculate the total number of tyres (columns)
total_tyres = df[['Left Front Tyre', 'Right Front Tyre', 'Left Rear Tyre', 'Right Rear Tyre', 'Spare Tyre']].shape[1]

# Calculate the percentage of 'OK' values
df['tyre_health_pct'] = (df['Tyre_Health'] / total_tyres) * 100


df.drop(columns=['Left Front Tyre', 'Right Front Tyre', 'Left Rear Tyre', 'Right Rear Tyre', 'Spare Tyre', 'Tyre_Health'], inplace=True)


df['ABSAntilockBrakingSystem'] = df['ABSAntilockBrakingSystem'].replace('not available', 0)

In [8]:
## check & drop based on corealtion
# Calculate the correlation matrix
correlation_matrix = df[['MaxPowerbhp', 'MaxPowerrpm', 'MaxTorqueNm']].corr()

# Set a threshold for correlation
threshold = 0.9

# Identify columns to drop
columns_to_drop = set()

# Loop through the correlation matrix to find pairs with correlation above the threshold
for i in range(len(correlation_matrix.columns)):
    for j in range(i):
        if abs(correlation_matrix.iloc[i, j]) > threshold:
            colname = correlation_matrix.columns[i]
            columns_to_drop.add(colname)

# Drop the identified columns from the DataFrame
df.drop(columns=columns_to_drop, inplace=True)

df = df[['ABSAntilockBrakingSystem','AirConditioner','Airbags','Bootspacelitres','Displacementcc','FueltankCapacitylitres','GroundClearancemm','tyre_health_pct','MaxPowerbhp','MaxPowerrpm','MaxTorqueNm','SeatingCapacity','content.bodyType','content.duplicateKey','content.fitnessUpto_months_remaining','content.fuelType','content.insuranceExpiry_months_remaining','content.insuranceType','content.make','content.odometerReading','content.ownerNumber','content.transmission','content.year','defects','repainted','content.onRoadPrice']]


In [9]:
### preprocessing

In [24]:
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
import pandas as pd

# Assuming your df is a DataFrame with the data

# Separate features (X) and target (y)
X = df.drop(columns=['content.onRoadPrice'])
y = df['content.onRoadPrice']

# Identify categorical columns (object type and binary 1/0 columns)
cat_cols = X.select_dtypes(include="object").columns.tolist()

# Add columns with exactly two unique values (0, 1) to categorical columns
binary_cols = [col for col in X.columns if set(X[col].dropna().unique()) == {0, 1}]
cat_cols.extend(binary_cols)

# Identify numerical columns (remaining columns after removing categorical ones)
num_cols = [col for col in X.columns if col not in cat_cols]

# Create separate pipelines for numeric and categorical transformations
num_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="mean")),  # Use mean imputation for numerical data
        ("scaler", StandardScaler())  # Standardize numerical features
    ]
)

cat_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),  # Use most frequent value for categorical imputation
        ("encoder", OneHotEncoder(sparse_output=False, drop='first'))  # One-hot encode, drop first to avoid multicollinearity
    ]
)

# Create the preprocessor using ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num_pipeline', num_pipeline, num_cols),  # Apply num_pipeline to numerical columns
        ('cat_pipeline', cat_pipeline, cat_cols)   # Apply cat_pipeline to categorical columns
    ]
)

# Split the data into train and test sets (be sure to pass y_train, not X_train, for the target variable)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Perform transformations on the training data
X_train_transformed = preprocessor.fit_transform(X_train)

# Apply the transformations to the test data
X_test_transformed = preprocessor.transform(X_test)

# Get the feature names after transformations
feature_names = preprocessor.get_feature_names_out()

# Convert the transformed data back to a DataFrame with the appropriate column names
X_train = pd.DataFrame(X_train_transformed, columns=feature_names)
X_test = pd.DataFrame(X_test_transformed, columns=feature_names)

In [25]:
X_train.head()

,num_pipeline__Airbags,num_pipeline__Bootspacelitres,num_pipeline__Displacementcc,num_pipeline__FueltankCapacitylitres,num_pipeline__GroundClearancemm,num_pipeline__tyre_health_pct,num_pipeline__MaxPowerbhp,num_pipeline__MaxPowerrpm,num_pipeline__MaxTorqueNm,num_pipeline__SeatingCapacity,num_pipeline__content.fitnessUpto_months_remaining,num_pipeline__content.insuranceExpiry_months_remaining,num_pipeline__content.odometerReading,num_pipeline__content.ownerNumber,num_pipeline__content.year,num_pipeline__defects,num_pipeline__repainted,cat_pipeline__content.bodyType_SUV,cat_pipeline__content.bodyType_Sedan,cat_pipeline__content.fuelType_Diesel,cat_pipeline__content.fuelType_Petrol,cat_pipeline__content.insuranceType_Comprehensive,cat_pipeline__content.insuranceType_Zero Depreciation,cat_pipeline__content.make_Ford,cat_pipeline__content.make_Honda,cat_pipeline__content.make_Hyundai,cat_pipeline__content.make_Jeep,cat_pipeline__content.make_KIA,cat_pipeline__content.make_MG,cat_pipeline__content.make_Mahindra,cat_pipeline__content.make_Maruti,cat_pipeline__content.make_Nissan,cat_pipeline__content.make_Renault,cat_pipeline__content.make_Skoda,cat_pipeline__content.make_Tata,cat_pipeline__content.make_Toyota,cat_pipeline__content.make_Volkswagen,cat_pipeline__content.transmission_Manual,cat_pipeline__ABSAntilockBrakingSystem_1.0,cat_pipeline__AirConditioner_1.0,cat_pipeline__content.duplicateKey_1
0,-0.147623,-0.924845,-0.957698,-0.814195,-0.876263,-1.740488,-0.958023,0.435627,-0.785420,-0.219943,-1.268176,-0.177121,0.216197,-0.541172,-1.199492,0.664376,-0.622443,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,0.000000,0.000000,-0.167914,0.170570,0.000000,0.250385,-0.347359,0.435627,-0.402939,-0.219943,-0.951374,0.390322,1.041933,1.636100,-0.851317,-0.202677,-0.310648,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0
2,-1.498785,0.000000,2.844377,2.263194,1.098435,0.914009,3.194495,-2.717131,3.538284,-0.219943,0.863037,1.525208,-0.324259,-0.541172,0.889556,-0.058168,-0.622443,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0
3,0.000000,-1.067615,-1.747482,-1.675864,0.586477,-0.413240,-1.527977,-0.015567,-1.084754,-0.219943,-0.116169,-0.460843,1.428421,-0.541172,-0.154968,-0.636203,1.560121,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
4,-0.147623,0.000000,-0.167914,0.170570,-0.876263,-1.076864,-0.347359,0.435627,-0.402939,-0.219943,-0.173770,-0.744564,1.733406,-0.541172,-0.154968,0.086341,1.248326,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0


In [26]:
### model training

In [33]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, Lasso, Ridge, ElasticNet
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.model_selection import cross_val_score
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from sklearn.model_selection import train_test_split

# Load your dataset (Assuming your dataset is preprocessed and ready)
# X, y = load_data()  # Replace with actual data loading step

# For demonstration, we assume X_train, X_test, y_train, y_test are already split and preprocessed.

# Models dictionary
models = {
    'LinearRegression': LinearRegression(),
    'Lasso': Lasso(),
    'Ridge': Ridge(),
    'ElasticNet': ElasticNet(),
    'RandomForest': RandomForestRegressor(),
    'XGBoost': XGBRegressor(),
    'SVR': SVR(),
    'KNN': KNeighborsRegressor(),
    'DecisionTree': DecisionTreeRegressor()
}

# Function to calculate adjusted R²
def adjusted_r2_score(r2, n, p):
    return 1 - (1 - r2) * (n - 1) / (n - p - 1)

# Function to evaluate model performance
def evaluate_model(true, pred):
    r2 = r2_score(true, pred)
    mae = mean_absolute_error(true, pred)
    mse = mean_squared_error(true, pred)
    return mae, mse, r2

# Define a function for deep learning model (Sequential NN)
def create_deep_learning_model(input_dim):
    model = Sequential()
    model.add(Dense(128, input_dim=input_dim, activation='relu'))
    model.add(Dense(64, activation='relu'))
    model.add(Dense(1))  # Regression output
    model.compile(optimizer='adam', loss='mse')
    return model

# Initialize lists to track results
trained_model_list = []
model_list = []
r2_list = []
adjusted_r2_list = []

# Variables to track the best model
best_model_name = None
best_model = None
best_adjusted_r2 = -np.inf  # Start with a very low value to ensure the first model is better

# Loop over models
for name, model in models.items():
    print(f"Training model: {name}")
    
    # Train the model
    model.fit(X_train, y_train)
    
    # Make predictions
    y_pred = model.predict(X_test)
    
    # Evaluate performance
    MAE, MSE, R2 = evaluate_model(y_test, y_pred)
    
    # Compute adjusted R²
    adjusted_r2 = adjusted_r2_score(R2, len(y_test), X_test.shape[1])
    
    # Print results for each model
    print(f"Model: {name}")
    print(f"MSE: {MSE}")
    print(f"MAE: {MAE}")
    print(f"R2: {R2}")
    print(f"Adjusted R2: {adjusted_r2}")
    
    # Track model with the best adjusted R²
    if adjusted_r2 > best_adjusted_r2:
        best_adjusted_r2 = adjusted_r2
        best_model_name = name
        best_model = model
    
    r2_list.append(R2)
    adjusted_r2_list.append(adjusted_r2)
    
    print("="*40)

    # Perform cross-validation
    cross_val_r2 = cross_val_score(model, X_train, y_train, scoring='r2', cv=5)
    print(f"Cross-validation R² scores: {cross_val_r2}")
    print(f"Mean CV R²: {cross_val_r2.mean()}")
    print("="*40)

# Deep Learning Model (Keras)
print("Training Deep Learning Model...")

# Create and compile deep learning model
input_dim = X_train.shape[1]
dl_model = create_deep_learning_model(input_dim)

# Train deep learning model
dl_model.fit(X_train, y_train, epochs=50, batch_size=32, verbose=0)

# Make predictions
dl_y_pred = dl_model.predict(X_test).flatten()

# Evaluate deep learning model
dl_MAE, dl_MSE, dl_R2 = evaluate_model(y_test, dl_y_pred)

# Calculate adjusted R² for deep learning
dl_adjusted_r2 = adjusted_r2_score(dl_R2, len(y_test), X_test.shape[1])

# Print deep learning model results
print("Deep Learning Model Results")
print(f"MSE: {dl_MSE}")
print(f"MAE: {dl_MAE}")
print(f"R2: {dl_R2}")
print(f"Adjusted R2: {dl_adjusted_r2}")
print("="*40)

# Update best model for deep learning if needed
if dl_adjusted_r2 > best_adjusted_r2:
    best_adjusted_r2 = dl_adjusted_r2
    best_model_name = "Deep Learning"
    best_model = dl_model

# Final summary of all models
print("Summary of Model Performances:")
for i, model_name in enumerate(models.keys()):
    print(f"{model_name}: R² = {r2_list[i]}, Adjusted R² = {adjusted_r2_list[i]}")

# Final for deep learning model
print(f"Deep Learning Model: R² = {dl_R2}, Adjusted R² = {dl_adjusted_r2}")

# Print the best model
print("\nBest Model based on Adjusted R²:")
print(f"Model: {best_model_name}")
print(f"Adjusted R²: {best_adjusted_r2}")
print(f"R²: {r2_score(y_test, best_model.predict(X_test))}")


Training model: LinearRegression
Model: LinearRegression
MSE: 18294337812.309933
MAE: 90990.97288256806
R2: 0.8943971405974779
Adjusted R2: 0.8888031906808068
Cross-validation R² scores: [0.88927445 0.9274861  0.91463833 0.91681569 0.91075235]
Mean CV R²: 0.9117933829125248
Training model: Lasso
Model: Lasso
MSE: 18293405660.175655
MAE: 90988.53941221342
R2: 0.894402521384243
Adjusted R2: 0.8888088564963282
Cross-validation R² scores: [0.88927885 0.92748815 0.9146467  0.91682243 0.91075922]
Mean CV R²: 0.91179907046784
Training model: Ridge
Model: Ridge
MSE: 18292462685.732132
MAE: 90980.28134303303
R2: 0.8944079646420751
Adjusted R2: 0.8888145880921074
Cross-validation R² scores: [0.8901503  0.92760962 0.91492052 0.91697768 0.91087506]
Mean CV R²: 0.9121066350788525
Training model: ElasticNet
Model: ElasticNet
MSE: 24834850427.716183
MAE: 107602.9247030981
R2: 0.8566424625527546
Adjusted R2: 0.8490485878300968
Cross-validation R² scores: [0.86426617 0.89252216 0.88237184 0.87609474 0.

In [34]:
## we will use RandomForest in production